#### SET UP

In [ ]:
import sqlite3

con = sqlite3.connect("chicago_analysis.db")
%load_ext sql
%sql sqlite:///chicago_analysis.db

## Crime Analysis

#### 1. Which community areas have the most crimes? (Top 10)

In [ ]:
%%sql
WITH area_names AS (
    SELECT DISTINCT community_area_number, community_area_name
    FROM SCHOOLS_DATA
)

SELECT 
    a.community_area_name AS "Community Area",
    PRINTF('%,d', COUNT(c.id)) AS "Number of Crimes"
FROM area_names a
JOIN CRIME_DATA c
    ON a.community_area_number = CAST(c.community_area AS INTEGER)
WHERE c.community_area IS NOT NULL
GROUP BY a.community_area_number
ORDER BY COUNT(c.id) DESC
LIMIT 10

##### Data Quality Investigation Queries*
###### *Refer to README file

In [ ]:
%sql SELECT community_area FROM CRIME_DATA ORDER BY community_area NULLS FIRST LIMIT 100 OFFSET 100

In [ ]:
%sql SELECT COUNT(*) - COUNT(community_area) AS null_count FROM CRIME_DATA

In [ ]:
%sql SELECT community_area, COUNT(*) AS crimes_count FROM CRIME_DATA GROUP BY community_area

In [ ]:

%%sql
SELECT SUM(crime_count) AS grouped_total
FROM (
    SELECT 
        community_area,
        COUNT(*) AS crime_count
    FROM CRIME_DATA
    WHERE community_area IS NOT NULL
    GROUP BY community_area
)

In [ ]:
%%sql

SELECT DISTINCT c.community_area
FROM CRIME_DATA c
LEFT JOIN SCHOOLS_DATA s
    ON CAST(c.community_area AS INTEGER) = s.community_area_number
WHERE s.community_area_number IS NULL
AND c.community_area IS NOT NULL
ORDER BY c.community_area

In [ ]:
%%sql

SELECT COUNT(*) AS unmatched_crimes
FROM CRIME_DATA c
LEFT JOIN SCHOOLS_DATA s
    ON CAST(c.community_area AS INTEGER) = s.community_area_number
WHERE s.community_area_number IS NULL
AND c.community_area IS NOT NULL

#### 2. What are the top 5 most common crime types?

In [ ]:
%%sql 
SELECT primary_type AS crime_type, PRINTF('%,d', COUNT(*)) AS crimes_count 
FROM CRIME_DATA 
GROUP BY primary_type
ORDER BY COUNT(*) DESC
LIMIT 5

#### 3. What % of crimes result in arrest?

In [ ]:
%%sql 
WITH crimes AS (
    SELECT COUNT(*) AS total_crimes,
    FROM CRIME_DATA)
SELECT ROUND(((COUNT(*) * 100.0) / c.total_crimes), 2) AS percentage_crimes_arrests 
FROM crimes c, CRIME_DATA
WHERE arrest = 1

#### 4. Which locations (location_description) are have the most crimes? (Top 5)

In [ ]:
%%sql 
SELECT 
    location_description AS dangerous_locations,
    COUNT(*) AS crime_count
FROM CRIME_DATA 
WHERE location_description IS NOT NULL
GROUP BY location_description
ORDER BY crime_count DESC
LIMIT 5

#### 5. Are crimes more domestic in high hardship areas?

In [ ]:
%%sql 

WITH area_names AS (
    SELECT DISTINCT community_area_number, community_area_name
    FROM SCHOOLS_DATA
)

SELECT 
    a.community_area_name AS "Community Area",
    PRINTF('%,d', COUNT(c.id)) AS "Number of Crimes",
    cd.hardship_index AS "Hardship Index"
FROM area_names a
JOIN CRIME_DATA c
    ON a.community_area_number = CAST(c.community_area AS INTEGER)
JOIN CENSUS_DATA cd 
    ON c.community_area = cd.ca
WHERE c.community_area IS NOT NULL
AND UPPER(c.description) LIKE '%DOMESTIC%'
GROUP BY a.community_area_number
ORDER BY COUNT(c.id) DESC
LIMIT 10

###### I Don't see a direct correlation, here we are showing the top 10 areas with the most crimes and the hardship indexes go up and down, while 60% of them are on the higher side the rest are not, there isn't a set pattern for this hypothesis.

## School Analysis

#### 1. What is the average safety score across all schools?

#### 2. Which school types (Elementary/Middle/High) perform best?

####  3. Top 10 schools by college enrollment

#### 4. Schools with safety score below average

#### 5. Is there a pattern between attendance and safety score?